# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranahassan9/Machine-Learning-FlyRank-AI/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Before testing any signal, we inspect distribution shape and sample size. This dataset has heavy-tailed traffic fields, so raw averages can be misleading.

In this section we print quantiles and skewness for key fields and log-scale summaries for impressions/sessions. We also print the base decline rate to anchor later tests.


In [ ]:
import numpy as np
import pandas as pd

# Load starter dataset used across Week 4 notebooks.
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Build decline label for audit only.
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

key_cols = [
    "impressions_90d", "clicks_90d", "sessions_90d",
    "days_since_last_update", "ctr", "avg_position", "ai_sessions_90d"
]

# Show heavy-tail evidence with quantiles and skewness.
summary = df[key_cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T[["count", "mean", "50%", "90%", "95%", "99%", "max"]]
summary["skew"] = df[key_cols].skew(numeric_only=True)
print("Distribution summary (look at 99% vs max and skew):")
print(summary.round(3).to_string())

# Log-transformed summaries for traffic columns to reduce heavy-tail dominance.
for col in ["impressions_90d", "sessions_90d", "clicks_90d"]:
    df[f"log1p_{col}"] = np.log1p(df[col].fillna(0))

log_cols = ["log1p_impressions_90d", "log1p_sessions_90d", "log1p_clicks_90d"]
log_summary = df[log_cols].describe(percentiles=[0.5, 0.9, 0.99]).T[["mean", "50%", "90%", "99%", "max"]]
print("\nLog-scale summary for key traffic columns:")
print(log_summary.round(3).to_string())

base_rate = df["is_declining_label"].mean()
print(f"\nBase decline rate (label prevalence): {base_rate:.4f}")


Distribution summary (look at 99% vs max and skew):
                          count      mean     50%       90%       95%        99%       max    skew
impressions_90d         30000.0  5200.366  731.00  12136.40  22996.50  73505.830  517715.0  11.385
clicks_90d              30000.0    16.097    1.00     32.00     69.05    253.010    4178.0  18.346
sessions_90d            30000.0    37.067    7.00     88.00    166.00    451.010    4345.0  12.127
days_since_last_update  30000.0    46.098   20.00    104.00    104.00    106.000     373.0   1.161
ctr                     30000.0     0.511    0.07      0.65      1.09      8.330     100.0  17.444
avg_position            30000.0    16.342   10.80     36.80     48.20     69.901     245.0   1.984
ai_sessions_90d         30000.0     0.204    0.00      0.00      1.00      5.000      64.0  19.136


KeyError: "['90%', '99%'] not in index"

## 2. Signal test #1 / #2 / #3 (verdict each)

Each signal gets the same structure:
- hypothesis
- bucketed table with `n`
- one-word verdict

Verdict labels follow course guidance: `CONFIRMED`, `OPPOSITE`, `MIXED`, `FALSE`.


In [2]:
# ---------- Signal 1: Staleness vs decline ----------
# Hypothesis: older pages are more likely to decline.
staleness_bins = [-1, 30, 90, 180, 365, np.inf]
staleness_labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]

audit = df.copy()
audit["days_since_last_update"] = audit["days_since_last_update"].fillna(audit["days_since_last_update"].median())
audit["staleness_bucket"] = pd.cut(audit["days_since_last_update"], bins=staleness_bins, labels=staleness_labels)

signal1 = (
    audit.groupby("staleness_bucket", observed=False)
         .agg(n=("content_id", "count"), decline_rate=("is_declining_label", "mean"))
         .reset_index()
)
signal1["decline_rate"] = signal1["decline_rate"].round(4)

# Require enough rows in every bucket before a strong claim.
s1_has_floor = bool((signal1["n"] >= 50).all())
s1_monotonic = signal1["decline_rate"].tolist() == sorted(signal1["decline_rate"].tolist())
s1_verdict = "CONFIRMED" if s1_has_floor and s1_monotonic else "MIXED"

print("Signal 1: Staleness vs decline")
print(signal1.to_string(index=False))
print("Verdict 1:", s1_verdict)

# ---------- Signal 2: CTR-position quality vs decline ----------
# Hypothesis: low CTR and weak rank buckets decline more.
visible = audit[(audit["impressions_90d"] >= 500) & (audit["avg_position"] > 0)].copy()
visible["ctr_bucket"] = np.where(visible["ctr"] < 0.03, "low_ctr", "ok_ctr")
visible["rank_bucket"] = np.where(visible["avg_position"] >= 10, "rank_10plus", "rank_top10")
visible["ctr_rank_bucket"] = visible["ctr_bucket"] + "__" + visible["rank_bucket"]

signal2 = (
    visible.groupby("ctr_rank_bucket")
           .agg(n=("content_id", "count"), decline_rate=("is_declining_label", "mean"))
           .reset_index()
           .sort_values("decline_rate", ascending=False)
)
signal2["decline_rate"] = signal2["decline_rate"].round(4)

s2_has_floor = bool((signal2["n"] >= 50).all())
s2_verdict = "CONFIRMED" if s2_has_floor and signal2.iloc[0]["decline_rate"] > signal2.iloc[-1]["decline_rate"] else "MIXED"

print("\nSignal 2: CTR-position bucket vs decline")
print(signal2.to_string(index=False))
print("Verdict 2:", s2_verdict)

# ---------- Signal 3: AI-active pages vs decline ----------
# Hypothesis: pages with AI sessions are less likely to be in decline (protection effect).
audit["ai_active"] = np.where(audit["ai_sessions_90d"] > 0, "ai_active", "no_ai")
signal3 = (
    audit.groupby("ai_active")
         .agg(n=("content_id", "count"), decline_rate=("is_declining_label", "mean"))
         .reset_index()
)
signal3["decline_rate"] = signal3["decline_rate"].round(4)

if (signal3["n"] >= 50).all():
    decline_ai = float(signal3.loc[signal3["ai_active"] == "ai_active", "decline_rate"].iloc[0])
    decline_no_ai = float(signal3.loc[signal3["ai_active"] == "no_ai", "decline_rate"].iloc[0])
    if decline_ai < decline_no_ai:
        s3_verdict = "CONFIRMED"
    elif decline_ai > decline_no_ai:
        s3_verdict = "OPPOSITE"
    else:
        s3_verdict = "MIXED"
else:
    s3_verdict = "FALSE"

print("\nSignal 3: AI-active vs decline")
print(signal3.to_string(index=False))
print("Verdict 3:", s3_verdict)

signal_verdicts = {
    "signal_1_staleness": s1_verdict,
    "signal_2_ctr_rank": s2_verdict,
    "signal_3_ai_protection": s3_verdict,
}
print("\nVerdict summary:", signal_verdicts)


Signal 1: Staleness vs decline
staleness_bucket     n  decline_rate
           0-30d 20480        0.5114
          31-90d   175        0.5886
         91-180d  9171        0.6111
        181-365d   169        0.4675
           365d+     5        0.6000
Verdict 1: MIXED

Signal 2: CTR-position bucket vs decline
     ctr_rank_bucket    n  decline_rate
 low_ctr__rank_top10  606        0.8185
 ok_ctr__rank_10plus 7068        0.6034
low_ctr__rank_10plus 2178        0.5927
  ok_ctr__rank_top10 6874        0.5687
Verdict 2: CONFIRMED

Signal 3: AI-active vs decline
ai_active     n  decline_rate
ai_active  1930        0.5528
    no_ai 28070        0.5413
Verdict 3: OPPOSITE

Verdict summary: {'signal_1_staleness': 'MIXED', 'signal_2_ctr_rank': 'CONFIRMED', 'signal_3_ai_protection': 'OPPOSITE'}


## 3. The flag-linked test

This test is directly linked to FlyRank-style refresh flags.

Flag logic tested:
- stale page (`days_since_last_update >= 180`)
- visible page (`impressions_90d >= 500`)

Question: do pages that match this rule show a meaningfully higher decline rate than non-flagged pages?


In [3]:
# Build one explicit flag candidate used in baseline-style operations.
audit["flag_stale_and_visible"] = (
    (audit["days_since_last_update"] >= 180) &
    (audit["impressions_90d"] >= 500)
).astype(int)

flag_test = (
    audit.groupby("flag_stale_and_visible")
         .agg(
             n=("content_id", "count"),
             decline_rate=("is_declining_label", "mean"),
             median_impressions_90d=("impressions_90d", "median"),
             median_days_since_update=("days_since_last_update", "median")
         )
         .reset_index()
)
flag_test["decline_rate"] = flag_test["decline_rate"].round(4)
flag_test["flag_label"] = flag_test["flag_stale_and_visible"].map({0: "not_flagged", 1: "flagged"})

print("Flag-linked test: stale + visible")
print(flag_test[["flag_label", "n", "decline_rate", "median_impressions_90d", "median_days_since_update"]].to_string(index=False))

flagged_rate = float(flag_test.loc[flag_test["flag_stale_and_visible"] == 1, "decline_rate"].iloc[0])
non_flagged_rate = float(flag_test.loc[flag_test["flag_stale_and_visible"] == 0, "decline_rate"].iloc[0])
rate_lift = flagged_rate - non_flagged_rate

if rate_lift > 0.03:
    flag_verdict = "CONFIRMED"
elif rate_lift < -0.03:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

print(f"\nFlag-linked verdict: {flag_verdict}")
print(f"Decline-rate lift (flagged - not_flagged): {rate_lift:.4f}")


Flag-linked test: stale + visible
 flag_label     n  decline_rate  median_impressions_90d  median_days_since_update
not_flagged 29983        0.5418                   731.0                      20.0
    flagged    17        0.9412                  4429.0                     194.0

Flag-linked verdict: CONFIRMED
Decline-rate lift (flagged - not_flagged): 0.3994


## 4. What this means in practice

This section translates the audit into action for a content team.

The summary below is generated from measured results, not assumptions.


In [4]:
# Build short decision-support notes from the measured verdicts.
insight_1 = (
    f"Signal verdicts: staleness={signal_verdicts['signal_1_staleness']}, "
    f"ctr_rank={signal_verdicts['signal_2_ctr_rank']}, ai_protection={signal_verdicts['signal_3_ai_protection']}."
)

insight_2 = (
    f"The stale+visible flag test is {flag_verdict} with a decline-rate lift of {rate_lift:.4f}, "
    "so this flag is directionally useful for triage when paired with quality checks."
)

insight_3 = (
    "In practice: prioritize high-visibility stale pages first, treat low-sample or contradictory buckets as review-only, "
    "and keep all rules leakage-safe by excluding label-derived columns like trend_direction and trend_pct."
)

print("Practice summary:")
print("1)", insight_1)
print("2)", insight_2)
print("3)", insight_3)


Practice summary:
1) Signal verdicts: staleness=MIXED, ctr_rank=CONFIRMED, ai_protection=OPPOSITE.
2) The stale+visible flag test is CONFIRMED with a decline-rate lift of 0.3994, so this flag is directionally useful for triage when paired with quality checks.
3) In practice: prioritize high-visibility stale pages first, treat low-sample or contradictory buckets as review-only, and keep all rules leakage-safe by excluding label-derived columns like trend_direction and trend_pct.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.